# Heart Failure Prediction

This notebook provides a reproducible, leakage-aware baseline for the UCI-style heart-failure classification dataset. It does **not** ship with patient data and does not claim clinical validity.

Run cells from top to bottom after setting `HEART_FAILURE_DATA` to `heart.csv` (or placing the file at `data/heart.csv`).

In [ ]:
import json
import os
import pickle
from pathlib import Path

import pandas as pd
from heart_failure_schema import (
    CATEGORICAL_COLUMNS,
    NUMERIC_COLUMNS,
    TARGET_COLUMN,
    validate_dataset,
)
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TARGET = TARGET_COLUMN
DATA_PATH = Path(os.environ.get("HEART_FAILURE_DATA", "data/heart.csv")).expanduser()

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Heart-failure dataset not found. Set HEART_FAILURE_DATA to the path "
        "of heart.csv, or place it at data/heart.csv. "
        f"Resolved path: {DATA_PATH.resolve()}"
    )

try:
    df = validate_dataset(pd.read_csv(DATA_PATH))
except (pd.errors.ParserError, UnicodeDecodeError) as exc:
    raise ValueError(f"Could not parse dataset CSV at {DATA_PATH}: {exc}") from exc
except (TypeError, ValueError) as exc:
    raise ValueError(f"Invalid heart-failure dataset schema: {exc}") from exc

print(f"Loaded {len(df):,} rows from {DATA_PATH}")
print("Class counts:")
print(df[TARGET].value_counts().sort_index())
df.head()


## Data checks

The preprocessing below is fitted **inside each model pipeline using only the training partition**. Categorical values are one-hot encoded with `handle_unknown="ignore"`; numeric values are median-imputed and standardized. No quantile bins or encoders are fitted on the full dataset.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# 70% train, 15% validation, 15% test; stratification preserves class balance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15 / 0.85,
    random_state=RANDOM_STATE, stratify=y_train
)
print({
    "train": X_train.shape,
    "validation": X_val.shape,
    "test": X_test.shape,
})
print("Train class proportions:")
print(y_train.value_counts(normalize=True).sort_index())

numeric_features = list(NUMERIC_COLUMNS)
categorical_features = list(CATEGORICAL_COLUMNS)
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            categorical_features,
        ),
    ],
    remainder="drop",
)


In [ ]:
model_specs = {
    "logistic_regression": LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE
    ),
    "decision_tree": DecisionTreeClassifier(
        max_depth=5, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=1
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.03, max_depth=3,
        random_state=RANDOM_STATE
    ),
    "svc": SVC(random_state=RANDOM_STATE),
    "knn": KNeighborsClassifier(n_neighbors=11),
    "gaussian_nb": GaussianNB(),
}

# XGBoost is optional at runtime so the notebook can still run without its
# native dependency. Install the commented requirement to add this candidate.
try:
    from xgboost import XGBClassifier
except ImportError:
    pass
else:
    model_specs["xgboost"] = XGBClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.03,
        subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_STATE, n_jobs=1, eval_metric="logloss",
    )


def make_model(estimator):
    return Pipeline([("preprocess", clone(preprocessor)), ("model", estimator)])


def calculate_metrics(labels, predicted):
    return {
        "accuracy": accuracy_score(labels, predicted),
        "precision": precision_score(labels, predicted, zero_division=0),
        "recall": recall_score(labels, predicted, zero_division=0),
        "f1": f1_score(labels, predicted, zero_division=0),
    }


def evaluate(model, features, labels):
    return calculate_metrics(labels, model.predict(features))


models = {name: make_model(estimator) for name, estimator in model_specs.items()}
validation_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    validation_rows.append({"model": name, **evaluate(model, X_val, y_val)})

validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values(["recall", "f1", "accuracy"], ascending=False)
    .reset_index(drop=True)
)
print(validation_results.to_string(index=False))


## Model selection and final holdout

Model selection uses validation recall (then F1 and accuracy as tie-breakers). The test set is held out until this point and is evaluated only for the selected model. Therefore, no test-set leaderboard is used for model selection. The reported values below are produced only when the real dataset is available; no results are committed to this repository.

In [ ]:
best_name = validation_results.loc[0, "model"]
# Refit only the selected pipeline on train + validation; the test set remains untouched.
X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)
final_model = clone(models[best_name])
final_model.fit(X_train_final, y_train_final)
test_predictions = final_model.predict(X_test)
test_metrics = calculate_metrics(y_test, test_predictions)

print(f"Selected model by validation recall: {best_name}")
print("Test metrics (single final holdout evaluation):")
print(pd.Series(test_metrics).to_frame("value"))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_predictions))
print(classification_report(y_test, test_predictions, zero_division=0))


In [ ]:
# Optional local artifact; artifacts/ is ignored by git.
ARTIFACT_DIR = Path(
    os.environ.get("HEART_FAILURE_ARTIFACT_DIR", "artifacts")
).expanduser()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / "best_model.pkl"
metadata_path = ARTIFACT_DIR / "best_model_metadata.json"
with model_path.open("wb") as handle:
    pickle.dump(final_model, handle, protocol=pickle.HIGHEST_PROTOCOL)
metadata = {
    "model": best_name,
    "target": TARGET,
    "random_state": RANDOM_STATE,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "test_metrics": {name: float(value) for name, value in test_metrics.items()},
}
metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(f"Saved selected model to {model_path.resolve()}")
print(f"Saved model metadata to {metadata_path.resolve()}")


## Reproducibility notes

- This workflow fixes the project seed at `42` and uses stratified train/validation/test splits.
- Cross-validation is not used for the baseline model comparison; if tuning is added, use `StratifiedKFold(shuffle=True, random_state=42)` and fit preprocessing inside the searched pipeline.
- Metrics are dataset- and version-dependent and are not medical evidence. The test set is small in the common source dataset, so uncertainty should be reported for any serious use.
